# Your Details

Your Name:

Your ID Number:

# Etivity Task 4 - Part 2: Quantizing a TensorFlow/Keras Model

For this exercise, you will apply various quantization strategies to a convolutional neural network (CNN) trained on the Fashion MNIST dataset. The first section of this exercise is already completed (Sections 1 and 2). Your task is to perform various quantizations on this model uses the TF Model optimisations toolkit and report on the results with your own code in Sections 3, 4 and 5.

By the end of this notebook, you'll be able to: 

* Understand Quantizations in TensorFlow 
* Quantize a CNN using the TensorFlow Model optimisation framework
* Analyse the model perfromance
* Results analysis

**Start** with sections [1] and [2] for which code is provided - then proceed with sections [3], [4] and [5] to begin this model quantization exercise.

    [1] Import data dependencies
    [2] Generate a TensorFlow/keras CNN model for the Fashion MNIST dataset
    [3] Convert model to TF Lite model
    [4] Perform Post Training Quantization (PTQ) to generate TF Lite model for:
        (a) PTQ using Float 16 Quantization
        (b) PTQ using Dynamic Range Quantization
        (c) PTQ using Full Integer (int8) Quantization 
        (d) Evaluate the TF Lite models
    [5] Perform Quantization Aware Training (QAT)
        (a) Train a TF model through tf.keras
        (b) Make it quantization-aware
        (c) Quantize the model using Dynamic Range Quantization
        (d) Evaluate the TF Lite model performance
    
   
### Important Note 1 on Submission 

There are code exercises to complete in this task.  Insert your code entries into the cell areas marked with the 'enter code here' text as below, so that grading can easily be assessed.

\### **ENTER CODE HERE**

Please make sure you are not doing the following:

1. You have not added any _extra_ `print` statement(s) in the assignment.
2. You have not added any _extra_ code cell(s) in the assignment.
3. You have not changed any of the function parameters.
4. You are not using any global variables inside your graded exercises. Unless specifically instructed to do so, please refrain from it and use the local variables instead.
5. You are not changing the assignment code where it is not required, like creating _extra_ variables.

### Important Note 2 on Submission 

There is a <font color='red'>**DISCUSSION**</font> section to include at the end of this notebook.

### Let's get started!

### Installing the TensorFlow Model Optimisation toolkit

You must first install it using pip (comment this out once you have done this).

<span style='color: red;'>**Note:**</span> There is no need to run this command again if used ok from the previous tutorial. (Hence commented out here)

In [37]:
# Install the TF optimization toolkit the first time 
! pip install -q tensorflow-model-optimization

## 1. Import the data dependencies

In [38]:
import numpy as np
import tensorflow as tf
import tensorflow 
import time
import os
import pathlib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow import keras

In [39]:
# Check that we are using a GPU
physical_devices = tf.config.experimental.list_physical_devices('GPU')
print("Num GPUs Available: ", len(physical_devices))

Num GPUs Available:  0


## 2. Generate a TensorFlow Model

We'll build a CNN model to classify the 10 fashion item categories from the [FASHION_MNIST dataset](https://www.tensorflow.org/datasets/catalog/fashion_mnist).

This training won't take long because you're training the model for just 5 epochs, which trains to about ~90% accuracy.

In [40]:
# Load Fashion MNIST dataset
fashion_mnist = tf.keras.datasets.fashion_mnist
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Reshape data for CNN input
img_width, img_height = 28, 28
X_train = X_train.reshape(X_train.shape[0], img_width, img_height, 1)
X_test = X_test.reshape(X_test.shape[0], img_width, img_height, 1)
input_shape = (img_width, img_height, 1)

# Normalize the input image so that each pixel value is between 0 to 1.
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0


# Define the model architecture
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Dropout(rate=0.1), # Randomly disable 10% of neurons
    tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Dropout(rate=0.1), # Randomly disable 10% of neurons
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])


# Build the model
model.compile(
    loss=tf.keras.losses.sparse_categorical_crossentropy, # loss function
    optimizer=tf.keras.optimizers.Adam(), # optimizer function
    metrics=['accuracy'] # reporting metric
)

# Train the fashion MNIST classification model
model.fit(
  X_train,
  y_train,
  epochs=5,
  validation_split=0.1
)

Epoch 1/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.7538 - loss: 0.6639 - val_accuracy: 0.8763 - val_loss: 0.3383
Epoch 2/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.8781 - loss: 0.3271 - val_accuracy: 0.8863 - val_loss: 0.3064
Epoch 3/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.8978 - loss: 0.2695 - val_accuracy: 0.9042 - val_loss: 0.2607
Epoch 4/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - accuracy: 0.9085 - loss: 0.2468 - val_accuracy: 0.9075 - val_loss: 0.2498
Epoch 5/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.9200 - loss: 0.2160 - val_accuracy: 0.9115 - val_loss: 0.2338


**Evaluate and save the model**

In [41]:
score = model.evaluate(X_test, y_test, verbose=1)
print("Test loss {:.4f}, accuracy {:.2f}%".format(score[0], score[1] * 100))

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9075 - loss: 0.2533
Test loss 0.2487, accuracy 90.84%


In [42]:
#Save the entire model into a model.h5 file
model.save("models/model.h5")
print("Saved model to disk")

Saved model to disk


## 3. Convert the trained model to TensorFlow Lite format

In the code cell below, convert the model to a **TensorFlow Lite** model and then save this unquantized TFLite model to the ./fashion_mnist_tflite_model directory

In [43]:
### ENTER CODE HERE
model = tf.keras.models.load_model("models/model.h5")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_models_dir = pathlib.Path("./fashion_tflite_models/")
tflite_models_dir.mkdir(exist_ok=True, parents=True)

tflite_model_file = tflite_models_dir / "fashion_model.tflite"
tflite_model_file.write_bytes(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\25287419\AppData\Local\Temp\tmp8og2zzv4\assets


INFO:tensorflow:Assets written to: C:\Users\25287419\AppData\Local\Temp\tmp8og2zzv4\assets


Saved artifact at 'C:\Users\25287419\AppData\Local\Temp\tmp8og2zzv4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  2693034021264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693034020304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693034019536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693034022992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693034022800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693034020688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693034019152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693034021072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693034021456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693034019920: TensorSpec(shape=(), dtype=tf.resource, name=None)


1825312

It's now a TensorFlow Lite model, but it's still using 32-bit float values for all parameter data.

## 4. Post-Training Quantization (PTQ)

### Part (a): PTQ using Float 16 Quantization
Here you will insert code for post-training float 16 quantization and then evaluate the file size compared to the unquantized tflite model size.

In [44]:
### ENTER CODE HERE
model = tf.keras.models.load_model("models/model.h5")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_quant16_model = converter.convert()

tflite_quant16_model_file = tflite_models_dir / "fashion_model_quant16.tflite"
tflite_quant16_model_file.write_bytes(tflite_quant16_model)


INFO:tensorflow:Assets written to: C:\Users\25287419\AppData\Local\Temp\tmpvjgqfc_7\assets


INFO:tensorflow:Assets written to: C:\Users\25287419\AppData\Local\Temp\tmpvjgqfc_7\assets


Saved artifact at 'C:\Users\25287419\AppData\Local\Temp\tmpvjgqfc_7'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  2693034019728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693036010064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693036010640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693036008720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693036008912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693036007568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693036007376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693036000272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693036006992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693036006608: TensorSpec(shape=(), dtype=tf.resource, name=None)


915740

**Evaluate the reduction in size of the model** - how much smaller is the Quantized 16-bit model?

In [45]:
### ENTER CODE HERE
print("Float model in Mb:", os.path.getsize(tflite_model_file) / float(2**20))
print("Quantized 16-bit model in Mb:", os.path.getsize(tflite_quant16_model_file) / float(2**20))
print("Compression ratio:", os.path.getsize(tflite_model_file)/os.path.getsize(tflite_quant16_model_file))

Float model in Mb: 1.740753173828125
Quantized 16-bit model in Mb: 0.8733177185058594
Compression ratio: 1.9932644637124075


### Part (b): PTQ using Dynamic Range Quantization
Next you will quantize the original model dynamically to change the model weight and activations from float to int8 format. Convert the model using **Dynamic Range Quantization** and evaluate the model file size reduction.

In [46]:
### ENTER CODE HERE
model = tf.keras.models.load_model("models/model.h5")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_quant_model = converter.convert()

tflite_quant_model_file = tflite_models_dir / "fashion_model_quant.tflite"
tflite_quant_model_file.write_bytes(tflite_quant_model)

print("Dynamic model size (MB):", os.path.getsize(tflite_quant_model_file) / (2**20))

INFO:tensorflow:Assets written to: C:\Users\25287419\AppData\Local\Temp\tmpyxsa7jzw\assets


INFO:tensorflow:Assets written to: C:\Users\25287419\AppData\Local\Temp\tmpyxsa7jzw\assets


Saved artifact at 'C:\Users\25287419\AppData\Local\Temp\tmpyxsa7jzw'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  2693036004112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595340048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595340624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595338704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595338896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595337552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595337360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595335824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595336976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595336400: TensorSpec(shape=(), dtype=tf.resource, name=None)
Dyna

 **Evaluate the reduction in size of the model** - how much smaller is the Quantized model?

In [47]:
### ENTER CODE HERE


float_size = os.path.getsize(tflite_model_file)/(2**20)
quant_size = os.path.getsize(tflite_quant_model_file)/(2**20)

print("Size reduction (MB):", float_size - quant_size)


interpreter = tf.lite.Interpreter(model_path=str(tflite_quant_model_file))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_shape = input_details[0]['shape']

acc = 0
for i in range(len(X_test)):
    input_data = X_test[i].reshape(input_shape)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()

    output_data = interpreter.get_tensor(output_details[0]['index'])

    if np.argmax(output_data) == y_test[i]:
        acc += 1

acc = acc / len(X_test)
print("Quantized model accuracy:", acc * 100)

Size reduction (MB): 1.2930450439453125
Quantized model accuracy: 90.84


### Part (c): PTQ using Full Integer (int8) Quantization 
Convert the original model to satisfy **full integer quantization** so that everything is converted (including activations) from float32 into int8 format. Evaluate the model file size reduction. Note you will need to use the OPTIMIZE_FOR_SIZE option by using a small representative dataset of the model and also make sure the input and output tensors are in int8 format.

In [48]:
### ENTER CODE HERE
def representative_data_gen():
    for input_value in tf.data.Dataset.from_tensor_slices(X_train).batch(1).take(100):
        yield [input_value]
model = tf.keras.models.load_model("models/model.h5")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_fullquant_model = converter.convert()

tflite_fullquant_model_file = tflite_models_dir / "fashion_model_fullquant.tflite"
tflite_fullquant_model_file.write_bytes(tflite_fullquant_model)

INFO:tensorflow:Assets written to: C:\Users\25287419\AppData\Local\Temp\tmpq54b5a3d\assets


INFO:tensorflow:Assets written to: C:\Users\25287419\AppData\Local\Temp\tmpq54b5a3d\assets


Saved artifact at 'C:\Users\25287419\AppData\Local\Temp\tmpq54b5a3d'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  2693036005840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2693036012944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595350608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595349456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595349648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595348304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595348112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595347344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595347728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2692595346192: TensorSpec(shape=(), dtype=tf.resource, name=None)


C:\Users\25287419\.conda\envs\cpu_env\Lib\site-packages\tensorflow\lite\python\convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


472576

**Check that the input and output tensors are in int8 format**

In [49]:
### ENTER CODE HERE

interpreter = tf.lite.Interpreter(model_content=tflite_fullquant_model)

print("Input dtype:", interpreter.get_input_details()[0]['dtype'])
print("Output dtype:", interpreter.get_output_details()[0]['dtype'])

Input dtype: <class 'numpy.uint8'>
Output dtype: <class 'numpy.uint8'>


 **Evaluate the reduction in size of the model** - how much smaller is the Quantized model?

In [50]:
### ENTER CODE HERE

full_quant_size = os.path.getsize(tflite_fullquant_model_file)/(2**20)

print("Size reduction (MB):", float_size - full_quant_size)

Size reduction (MB): 1.290069580078125


### Part (d):  Evaluate the TF Lite models on all images

In this section, evaluate the four TF Lite models by running inference using the TensorFlow Lite [`Interpreter`](https://www.tensorflow.org/api_docs/python/tf/lite/Interpreter) to compare the model accuracies. First, build a **run_tflite_model()** function to run inference on a TF Lite model and then an **evaluate_model()** function to evaluate the TF Lite model on all images in the X_test dataset.

**Evaluate the model performance for these models** by reporting on the model accuracies.
1. Float model (Unquantized)
2. 16-bit quantized model
3. Initial quantized 8-bit model
4. Fully quantized 8-bit model 

In [51]:
def run_tflite_model(tflite_file, X_test):
    interpreter = tf.lite.Interpreter(model_path=str(tflite_file))
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_shape = input_details['shape']

    predictions = []

    for i in range(len(X_test)):
        input_data = X_test[i]

       
        if input_details['dtype'] == np.uint8:
            scale, zero_point = input_details['quantization']
            input_data = input_data / scale + zero_point
            input_data = input_data.astype(np.uint8)

        input_data = np.expand_dims(input_data, axis=0)

        interpreter.set_tensor(input_details['index'], input_data)
        interpreter.invoke()

        output = interpreter.get_tensor(output_details['index'])[0]
        predictions.append(np.argmax(output))

    return np.array(predictions)

1. Evaluate the float model

In [52]:
### ENTER CODE HERE
def evaluate_model(tflite_file, X_test, y_test):
    preds = run_tflite_model(tflite_file, X_test)
    acc = np.mean(preds == y_test)
    return acc
print("Float model accuracy:", evaluate_model(tflite_model_file, X_test, y_test) * 100)

Float model accuracy: 90.84


2. Evaluate the 16-bit quantized model

In [53]:
### ENTER CODE HERE

print("Float16 model accuracy:", evaluate_model(tflite_quant16_model_file, X_test, y_test) * 100)

Float16 model accuracy: 90.84


3. Evaluate the initial quantized 8-bit model

In [54]:
### ENTER CODE HERE

print("Dynamic quant model accuracy:", evaluate_model(tflite_quant_model_file, X_test, y_test) * 100)

Dynamic quant model accuracy: 90.84


4. Evaluate the fully quantized 8-bit integer model

In [55]:
### ENTER CODE HERE

print("Full quant model accuracy:", evaluate_model(tflite_fullquant_model_file, X_test, y_test) * 100)

Full quant model accuracy: 90.86


## 5. Quantization-Aware Training (QAT)

QAT models quantization during training and typically provides higher accuracies as compared to post-training quantization. 
Generally, QAT is a three-step process:

    (a) Train a regular model through tf.keras 
        YOU MAY HAVE TO 'import tf_keras as keras' and use model = keras.Sequential([...]) format.
    (b) Make it quantization-aware by applying the related API, allowing it to learn those loss-robust parameters.
    (c) Quantize the model using one of the approaches mentioned above and analyse performance


### **Part (a)**: Train a model for the FASHION MNIST dataset again

In [56]:
### ENTER CODE HERE
import tf_keras as keras

model = keras.Sequential([
  keras.layers.InputLayer(input_shape=(28, 28)),
  keras.layers.Reshape(target_shape=(28, 28, 1)),
  keras.layers.Conv2D(filters=12, kernel_size=(3, 3), activation='relu'),
  keras.layers.MaxPooling2D(pool_size=(2, 2)),
  keras.layers.Flatten(),
  keras.layers.Dense(10)
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(X_train, y_train, epochs=5, validation_split=0.1)

Epoch 1/5
1688/1688 [==============================] - 4s 2ms/step - loss: 2.3457 - accuracy: 0.3594 - val_loss: 2.3026 - val_accuracy: 0.3617
Epoch 2/5
1688/1688 [==============================] - 3s 2ms/step - loss: 2.3026 - accuracy: 0.3592 - val_loss: 2.3026 - val_accuracy: 0.3617
Epoch 3/5
1688/1688 [==============================] - 3s 2ms/step - loss: 2.3026 - accuracy: 0.3592 - val_loss: 2.3026 - val_accuracy: 0.3617
Epoch 4/5
1688/1688 [==============================] - 3s 2ms/step - loss: 2.3026 - accuracy: 0.3592 - val_loss: 2.3026 - val_accuracy: 0.3617
Epoch 5/5
1688/1688 [==============================] - 3s 2ms/step - loss: 2.3026 - accuracy: 0.3592 - val_loss: 2.3026 - val_accuracy: 0.3617


### Part (b): Make the model quantization aware
Hint: Use q_aware_model = quantize_model(model)

In [57]:
### ENTER CODE HERE

import tensorflow_model_optimization as tfmot

q_aware_model = tfmot.quantization.keras.quantize_model(model)

#### Retrain the quantization aware model

In [58]:
### ENTER CODE HERE

q_aware_model.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

q_aware_model.fit(X_train, y_train, epochs=3, validation_split=0.1)

Epoch 1/3
1688/1688 [==============================] - 5s 2ms/step - loss: 2.3026 - accuracy: 0.3585 - val_loss: 2.3026 - val_accuracy: 0.3550
Epoch 2/3
1688/1688 [==============================] - 4s 2ms/step - loss: 2.3026 - accuracy: 0.3552 - val_loss: 2.3026 - val_accuracy: 0.3592
Epoch 3/3
1688/1688 [==============================] - 4s 2ms/step - loss: 2.3026 - accuracy: 0.3569 - val_loss: 2.3026 - val_accuracy: 0.3620


#### Compare the accuracy of the baseline model to the new QAT model

In [59]:
### ENTER CODE HERE

baseline_acc = model.evaluate(X_test, y_test, verbose=0)[1]
qat_acc = q_aware_model.evaluate(X_test, y_test, verbose=0)[1]

print("Baseline accuracy:", baseline_acc * 100)
print("QAT accuracy:", qat_acc * 100)

Baseline accuracy: 36.160001158714294
QAT accuracy: 36.10999882221222


#### Fine tune with QAT on a subset of the training data

In [60]:
### ENTER CODE HERE

q_aware_model.fit(X_train[:10000], y_train[:10000], epochs=2)

Epoch 1/2
313/313 [==============================] - 1s 2ms/step - loss: 2.3026 - accuracy: 0.3611
Epoch 2/2
313/313 [==============================] - 1s 2ms/step - loss: 2.3026 - accuracy: 0.3602


#### Re-evaluate the model accuracies.

In [61]:
### ENTER CODE HERE

qat_acc = q_aware_model.evaluate(X_test, y_test, verbose=0)[1]
print("Final QAT accuracy:", qat_acc * 100)

Final QAT accuracy: 35.87999939918518


#### Save the QAT model to the ./models directory

In [62]:
### ENTER CODE HERE

q_aware_model.save("models/qat_model.h5")

C:\Users\25287419\.conda\envs\cpu_env\Lib\site-packages\tf_keras\src\engine\training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


### Part (c): Convert the model to TF Lite format  using Dynamic Range Quantization

In [63]:
### ENTER CODE HERE

converter = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_qat_model = converter.convert()

tflite_qat_model_file = tflite_models_dir / "fashion_model_qat.tflite"
tflite_qat_model_file.write_bytes(tflite_qat_model)

INFO:tensorflow:Assets written to: C:\Users\25287419\AppData\Local\Temp\tmpwmkg8iw0\assets


INFO:tensorflow:Assets written to: C:\Users\25287419\AppData\Local\Temp\tmpwmkg8iw0\assets
C:\Users\25287419\.conda\envs\cpu_env\Lib\site-packages\tensorflow\lite\python\convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


24824

**Evaluate the reduction in size of the model.** 

In [64]:
### ENTER CODE HERE

qat_size = os.path.getsize(tflite_qat_model_file) / (2**20)
print("QAT model size (MB):", qat_size)
print("Size reduction (MB):", float_size - qat_size)

QAT model size (MB): 0.02367401123046875
Size reduction (MB): 1.7170791625976562


### Part (d): Evaluate the TF Lite QAT model accuracy
Hint: Use the intrepreter evaluate_model() function to get the accuracy result.

In [65]:
### ENTER CODE HERE

print("QAT TFLite accuracy:", evaluate_model(tflite_qat_model_file, X_test, y_test) * 100)

ValueError: Cannot set tensor: Dimension mismatch. Got 4 but expected 3 for input 0.

In [ ]:
### ENTER CODE HERE

acc = evaluate_model(tflite_qat_model_file, X_test, y_test)
print("Accuracy:", acc * 100)

## <span style='color: red;'>Comment on the results of this exercise:</span> ##


Add your final comments and observations here that match your results from your notebook exercise: